## 임포트 및 초기 설정

In [ ]:
import os, sys, random, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('.')
from src.utils import set_seed, make_run_dir
from src.visualization import (plot_loss_history, plot_weight_tradeoff,
                                plot_confusion_matrix, plot_model_comparison_pareto,
                                plot_model_comparison_bar, plot_gamma_sweep)
from src.engine import measure_inference_speed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0  # Windows/Jupyter에서는 0이 가장 안정적
print(f"Device: {device} | num_workers: {NUM_WORKERS}")

run_dirs = make_run_dir('./outputs')

Device: cuda | num_workers: 0
새로운 실험 결과 폴더가 생성되었습니다: ./outputs\run_20260401_183925


## 커스텀 Dataset 및 Focal Loss 정의

In [8]:
# ── 커스텀 Dataset (인덱스 기반, fold 분할에 필요)
class ScrewDataset(Dataset):
    def __init__(self, img_paths, labels, transform=None):
        self.img_paths = img_paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# ── Focal Loss (gamma sweep 대상)
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        """
        gamma : focusing parameter (클수록 어려운 샘플에 더 집중)
        alpha : 클래스별 가중치 텐서 (불균형 보정, 선택사항)
        """
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(logits, targets,
                                              weight=self.alpha,
                                              reduction='none')
        pt      = torch.exp(-ce_loss)                     # 예측 확신도
        focal   = ((1 - pt) ** self.gamma) * ce_loss      # 어려운 샘플 강조

        if self.reduction == 'mean':
            return focal.mean()
        return focal.sum()


# ── Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Dataset, FocalLoss, Transform 정의 완료")

Dataset, FocalLoss, Transform 정의 완료


## 데이터 로딩 (k-fold_data → 경로 + 라벨 리스트)

In [9]:
DATA_DIR    = './data/k-fold_data'
CLASS_NAMES = ['good', 'type1', 'type2', 'type3', 'type4', 'type5']

all_paths, all_labels = [], []

for label_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(DATA_DIR, class_name)
    for fname in sorted(os.listdir(class_dir)):
        if fname.lower().endswith('.png'):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(label_idx)

all_labels_np = np.array(all_labels)

# 분포 확인
print(f"전체 샘플: {len(all_paths)}장")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {(all_labels_np == i).sum()}장")

전체 샘플: 480장
  good: 361장
  type1: 24장
  type2: 24장
  type3: 25장
  type4: 23장
  type5: 23장


## Stratified 5-Fold + WeightedRandomSampler + Focal Loss gamma sweep

In [10]:
def build_model_6cls(model_name):
    """num_classes=6 으로 헤드 교체"""
    if model_name == 'resnet18':
        m = models.resnet18(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, 6)
    elif model_name == 'mobilenet_v2':
        m = models.mobilenet_v2(weights='DEFAULT')
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 6)
    elif model_name == 'vgg16':
        m = models.vgg16(weights='DEFAULT')
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, 6)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return m


def f2_score(precision, recall):
    denom = 4 * precision + recall
    return 5 * precision * recall / denom if denom > 0 else 0.0


def run_kfold_experiment(model_name, all_paths, all_labels_np,
                         gamma_list, n_splits=5,
                         num_epochs=50, batch_size=16,
                         num_workers=NUM_WORKERS,
                         device=device, run_dirs=run_dirs):

    skf          = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    gamma_list   = list(gamma_list)

    # gamma별 fold 평균 F2 저장
    gamma_f2_map = {g: [] for g in gamma_list}

    print(f"\n{'='*60}")
    print(f"모델: {model_name.upper()} | {n_splits}-Fold CV | gamma sweep: {gamma_list}")
    print(f"{'='*60}")

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels_np)):
        print(f"\n[Fold {fold_idx+1}/{n_splits}]")

        train_labels_fold = all_labels_np[train_idx]

        # ── WeightedRandomSampler (train split에만 적용)
        class_counts  = np.bincount(train_labels_fold, minlength=6)
        class_weights = 1.0 / np.where(class_counts == 0, 1, class_counts)
        sample_w      = torch.tensor(class_weights[train_labels_fold], dtype=torch.float)
        sampler       = WeightedRandomSampler(sample_w, num_samples=len(train_idx), replacement=True)

        train_paths_fold = [all_paths[i] for i in train_idx]
        val_paths_fold   = [all_paths[i] for i in val_idx]

        train_ds = ScrewDataset(train_paths_fold, train_labels_fold,            transform=train_transform)
        val_ds   = ScrewDataset(val_paths_fold,   all_labels_np[val_idx], transform=val_transform)

        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                                  num_workers=num_workers, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                                  num_workers=num_workers, pin_memory=True)

        # ── gamma sweep
        for gamma in gamma_list:
            set_seed(42)
            model     = build_model_6cls(model_name).to(device)
            criterion = FocalLoss(gamma=gamma)
            optimizer = optim.Adam(model.parameters(), lr=1e-4)
            scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

            best_val_loss   = float('inf')
            no_improve      = 0
            patience        = 7

            for epoch in range(num_epochs):
                # Train
                model.train()
                for imgs, lbls in train_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    optimizer.zero_grad()
                    loss = criterion(model(imgs), lbls)
                    loss.backward()
                    optimizer.step()
                scheduler.step()

                # Val loss (early stopping용)
                model.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for imgs, lbls in val_loader:
                        imgs, lbls = imgs.to(device), lbls.to(device)
                        val_loss += criterion(model(imgs), lbls).item()

                if (epoch + 1) % 10 == 0 or epoch == 0:
                    print(f"    [{model_name}][Fold {fold_idx+1}][gamma={gamma}] Epoch {epoch+1:02d}/{num_epochs} | val_loss={val_loss:.4f}")

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    no_improve    = 0
                else:
                    no_improve += 1
                if no_improve >= patience:
                    print(f"    -> early stop at epoch {epoch+1}")
                    break

            # Val 평가 (macro F2)
            model.eval()
            y_true, y_pred = [], []
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    preds = model(imgs.to(device)).argmax(dim=1).cpu()
                    y_true.extend(lbls.numpy())
                    y_pred.extend(preds.numpy())

            report = classification_report(y_true, y_pred, output_dict=True,
                                           zero_division=0)
            # 결함 클래스(type1~5)에 대한 macro F2
            defect_f2s = []
            for i in range(1, 6):  # type1~type5 (label 1~5)
                p = report.get(str(i), {}).get('precision', 0)
                r = report.get(str(i), {}).get('recall',    0)
                defect_f2s.append(f2_score(p, r))
            macro_f2 = np.mean(defect_f2s)

            gamma_f2_map[gamma].append(macro_f2)
            print(f"  gamma={gamma} | macro F2(결함): {macro_f2:.4f} (epoch {epoch+1}에 early stop)")

    # gamma별 평균 F2
    print(f"\n{'='*60}")
    print(f"{model_name.upper()} gamma sweep 결과 (5-fold 평균 macro F2)")
    best_gamma, best_avg_f2 = None, -1.0
    for g, scores in gamma_f2_map.items():
        avg = np.mean(scores)
        print(f"  gamma={g:.1f} → 평균 F2: {avg:.4f}")
        if avg > best_avg_f2:
            best_avg_f2, best_gamma = avg, g
    print(f"\n최적 gamma: {best_gamma}  (평균 F2: {best_avg_f2:.4f})")
    print(f"{'='*60}")

    return best_gamma, best_avg_f2, gamma_f2_map

## fps 측정 함수

In [ ]:
def train_final_model(model_name, best_gamma, all_paths, all_labels_np,
                      num_epochs=50, batch_size=16, num_workers=NUM_WORKERS, device=device):
    """best gamma로 전체 데이터 학습 후 FPS 측정"""
    set_seed(42)

    class_counts  = np.bincount(all_labels_np, minlength=6)
    class_weights = 1.0 / np.where(class_counts == 0, 1, class_counts)
    sample_w      = torch.tensor(class_weights[all_labels_np], dtype=torch.float)
    sampler       = WeightedRandomSampler(sample_w, num_samples=len(all_paths), replacement=True)

    full_ds     = ScrewDataset(all_paths, all_labels_np, transform=train_transform)
    full_loader = DataLoader(full_ds, batch_size=batch_size, sampler=sampler,
                             num_workers=num_workers, pin_memory=True)

    model     = build_model_6cls(model_name).to(device)
    criterion = FocalLoss(gamma=best_gamma)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    for epoch in range(num_epochs):
        model.train()
        for imgs, lbls in full_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            criterion(model(imgs), lbls).backward()
            optimizer.step()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"  [{model_name}] Epoch {epoch+1}/{num_epochs} 완료")

    fps, _ = measure_inference_speed(model, device)
    torch.save(model.state_dict(),
               os.path.join(run_dirs['weights'], f'{model_name}_final_gamma{best_gamma}.pth'))
    return model, fps

## 실험 실행1 ResNet

In [ ]:
GAMMA_LIST = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

In [ ]:
# ResNet-18
best_gamma_resnet, best_f2_resnet, gamma_map_resnet = run_kfold_experiment(
    'resnet18', all_paths, all_labels_np, GAMMA_LIST)

In [ ]:
plot_gamma_sweep(gamma_map_resnet, 'resnet18',     run_dirs['figures'])
model_resnet, fps_resnet = train_final_model('resnet18',     best_gamma_resnet, all_paths, all_labels_np)

## 실험 실행2 MobileNet

In [ ]:
# MobileNet-V2
best_gamma_mobile, best_f2_mobile, gamma_map_mobile = run_kfold_experiment(
    'mobilenet_v2', all_paths, all_labels_np, GAMMA_LIST)

In [ ]:
plot_gamma_sweep(gamma_map_mobile, 'mobilenet_v2', run_dirs['figures'])
model_mobile, fps_mobile = train_final_model('mobilenet_v2', best_gamma_mobile, all_paths, all_labels_np)

## 실험 실행3 VGG

In [ ]:
# VGG-16
best_gamma_vgg, best_f2_vgg, gamma_map_vgg = run_kfold_experiment(
    'vgg16', all_paths, all_labels_np, GAMMA_LIST)

In [ ]:
plot_gamma_sweep(gamma_map_vgg,    'vgg16',         run_dirs['figures'])
model_vgg,    fps_vgg    = train_final_model('vgg16',         best_gamma_vgg,    all_paths, all_labels_np)

## 최종 비교 차트

In [ ]:
ALPHA = 0.7  # 0.7 * F2 + 0.3 * FPS_norm

model_names = ['resnet18', 'mobilenet_v2', 'vgg16']
f2_scores   = [best_f2_resnet, best_f2_mobile, best_f2_vgg]
fps_list    = [fps_resnet,     fps_mobile,     fps_vgg]

plot_model_comparison_pareto(model_names, f2_scores, fps_list,
                             ALPHA, run_dirs['figures'])
plot_model_comparison_bar(model_names, f2_scores, fps_list,
                          ALPHA, run_dirs['figures'])

print("\n[ 최종 결과 요약 ]")
max_fps = max(fps_list)
for name, f2, fps in zip(model_names, f2_scores, fps_list):
    fps_norm = fps / max_fps
    e_score  = ALPHA * f2 + (1 - ALPHA) * fps_norm
    print(f"  {name:15s} | F2: {f2:.4f} | FPS: {fps:.1f} | E-Score: {e_score:.4f}")